In [1]:
import torch

In [2]:
# toy dataset

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

In [3]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

Class labels in PyTorch is required to start from 0. That is, if we are havivng two classes, the label should be 0 and 1. Also, the number of the output nodes be equal to the number of classes, or the highest label value + 1.

In [4]:
from torch.utils.data import Dataset # PyTorch Dataset class


class ToyDataset(Dataset):
    def __init__(self, X, y):
        """
        Sets up the data attributes that can be accessed using the class methods __getitem__ and __len__
        """
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        """
        Instruction to return exactly one item from the dataset using an index
        """
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        """
        To get the length of the dataset
        """
        return self.labels.shape[0]

In [5]:
from torch.utils.data import DataLoader # to sample from the dataset

torch.manual_seed(123)

# This helps in defining the PyTorch dataset
train_ds = ToyDataset(X_train, y_train)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)


test_ds = ToyDataset(X_test, y_test)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [9]:
# Iterate over the data loader just to show the tensor per batch
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [ 2.7000, -1.5000]]) tensor([1, 1])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.9000,  2.9000]]) tensor([0, 0])
Batch 3: tensor([[-0.5000,  2.6000]]) tensor([0])


The iteration is repeated to demonstrate a change in shuffling when the dataset is iterated the second time, even with the `torch.manual_seed`.
Also note that because we have a single element in the last batch because we have batch size to be 5-element datasets

In [10]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-0.5000,  2.6000],
        [ 2.7000, -1.5000]]) tensor([0, 1])
Batch 2: tensor([[-0.9000,  2.9000],
        [ 2.3000, -1.1000]]) tensor([0, 1])
Batch 3: tensor([[-1.2000,  3.1000]]) tensor([0])


In [11]:
# To avoid disturbing convergence on the last batch of a training epoch, set `drop_last` to True

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

# let's check
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-1.2000,  3.1000]]) tensor([1, 0])
Batch 2: tensor([[-0.5000,  2.6000],
        [ 2.7000, -1.5000]]) tensor([0, 1])


The `num_workers` need to be set to value greater than 0 so that multiple worker processes are launched to load the data in parrallel. This frees the main process to focus on model training and better system resource utilization

A typical training loop

In [12]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [13]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)

        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00
